# Implementação do Modelo - Fase 3

Refinamento da Implementação, análise dos resulados, ajustes dos hyperparâmetros

In [1]:
%load_ext autoreload
%autoreload 2


import datetime
import glob
import os
import random

from philder import infra
from philder import database_consolidation as consolidacao
from philder import data_processing as processamento
from philder import database_balance as balanceamento
from philder import performance_assessment as avaliacao


# --- Preparação ---
infra.criar_estrutura_diretorios()

# --- Módulo A: consolidação da base ---
#brutos_legitimos = consolidacao.importar_corpus_spamassassin("dados_philder/spamassassin")
#brutos_phishing = consolidacao.importar_corpus_nazario("dados_philder/Nazario.csv")
#df = consolidacao.consolidar_dataset(brutos_legitimos, brutos_phishing)

# --- Módulo B: processamento de texto ---
# df = processamento.pipeline_processamento(df)

# --- Módulo C: balanceamento + treino + teste ---
# df_balanceado = balanceamento.aplicar_undersampling(df)
# df_treino, df_teste = balanceamento.dividir_treino_teste(df_balanceado)
# tokenizer, modelo_tb = balanceamento.carregar_tokenizer_e_modelo()
# dataset_treino, dataset_teste = balanceamento.tokenizar_dataset(df_treino, df_teste, tokenizer)
# trainer, tempo_treino = balanceamento.treinar_modelo(modelo_tb, tokenizer, dataset_treino, dataset_teste)
# predicoes, tempo_teste = balanceamento.testar_modelo(trainer, dataset_teste)
# balanceamento.salvar_modelo(trainer, tokenizer)

# --- Módulo D: avaliação de desempenho ---
# metricas = avaliacao.calcular_metricas_primarias(df_teste["class_label"].tolist(), predicoes)
# eficiencia = avaliacao.calcular_metricas_eficiencia(metricas["accuracy"], tempo_treino, tempo_teste, len(df_teste))
# avaliacao.gerar_curva_perda(trainer.state.log_history)
# avaliacao.salvar_resultados({**metricas, **eficiencia})


Estrutura de diretórios criada/confirmada:
  - dados_philder/
  - modelo/  (com modelo\checkpoints/)
  - resultados/


In [2]:
# --- Fase 3 (01/10/2026): configuração do experimento (modelo + padrão de saída) ---
# Padrão de saída:
#   modelo/<modelo_escolhido>/               -> modelo final treinado
#   modelo/<modelo_escolhido>/checkpoints/   -> output_dir do Trainer
#   resultados/<modelo_escolhido>/resultado_final-<rodada>_<ddmmaaaa>.csv
#   resultados/<modelo_escolhido>/curva_perda-<rodada>_<ddmmaaaa>.png
# O número da rodada é calculado automaticamente (maior rodada do dia na pasta + 1).
modelos_artigo = {
    "tinybert":   "huawei-noah/TinyBERT_General_4L_312D",
    "minilm":     "microsoft/MiniLM-L12-H384-uncased",
    "distilbert": "distilbert-base-uncased",
    "mobilebert": "google/mobilebert-uncased",
    "albert":     "albert-base-v2",
}
modelo_escolhido = "albertt"            # <-- único ponto a trocar entre experimentos
checkpoint_modelo = modelos_artigo[modelo_escolhido]

# Alvo do undersampling por classe (0 = legítimo, 1 = phishing) — 07/10/2026
# {0: 502, 1: 501} -> fiel ao artigo (Figura 2: 1.003 registros)
# None             -> todas as classes no tamanho da minoritária (501 + 501, comportamento até 07/10)
undersampling_alvo = {0: 502, 1: 501}
data_execucao = datetime.date.today().strftime("%d%m%Y")

pasta_modelo = os.path.join("modelo", modelo_escolhido)
pasta_checkpoints = os.path.join(pasta_modelo, "checkpoints")
pasta_resultados = os.path.join("resultados", modelo_escolhido)


def proxima_rodada(pasta: str, data: str) -> int:
    """Próximo número de rodada do dia em `pasta` (1 se ainda não houver nenhuma)."""
    arquivos = glob.glob(os.path.join(pasta, f"resultado_final-*_{data}.csv"))
    numeros = [int(os.path.basename(a).split("-")[1].split("_")[0]) for a in arquivos]
    return max(numeros, default=0) + 1


print(f"Modelo: {modelo_escolhido} ({checkpoint_modelo}) | saída: {pasta_modelo}/ e {pasta_resultados}/ "
      f"| undersampling: {undersampling_alvo}")

Modelo: tinybert (huawei-noah/TinyBERT_General_4L_312D) | saída: modelo\tinybert/ e resultados\tinybert/ | undersampling: {0: 502, 1: 501}


# A) DATABASE CONSOLIDATION

<DESENVOLVER>

In [2]:
# --- Módulo A: consolidação da base ---
brutos_legitimos = consolidacao.importar_corpus_spamassassin("dados_philder/spamassassin")
brutos_phishing = consolidacao.importar_corpus_nazario("dados_philder/Nazario.csv")

# 19/09/2026 -  Ajuste da base para refletir a quantidade de amostras do artigo original, especifcamente
# da base do phishtank
# Nazario tem mais e-mails de phishing do que o PhishTank usado no artigo (501) —
# reduz por amostragem aleatória (sem reposição) para equiparar a mesma quantidade.
# SpamAssassin já está na mesma contagem do artigo (2.501), não precisa de ajuste.
## 25/09/2026 desativado pois foi incorporada na etapa de deduplicação
## random.seed(42)
## brutos_phishing = random.sample(brutos_phishing, 501)

legitimos_amostrados, phishing_amostrados = consolidacao.amostrar_quantidade_artigo(
    brutos_legitimos, brutos_phishing, deduplicar_antes=True
)

df = consolidacao.consolidar_dataset(legitimos_amostrados, phishing_amostrados)

## 25/09/2026 ajuste por implementação da deduplicação
## df = consolidacao.consolidar_dataset(brutos_legitimos, brutos_phishing)

Total de e-mails legítimos carregados: 2501
Total de e-mails de phishing carregados: 1565 (coluna: 'body')
Pré-processamento concluído: 1565 registros (38 ficaram vazios)
Deduplicação prévia do pool de 'phishing' (texto processado): 1565 e-mails -> 1471 únicos (94 duplicatas descartadas antes da amostragem)
Pré-processamento concluído: 2501 registros (2 ficaram vazios)
Deduplicação prévia do pool de 'legítimos' (texto processado): 2501 e-mails -> 2446 únicos (55 duplicatas descartadas antes da amostragem)
Aviso: pool único de 'legítimos' tem 2446 e-mails, igual ou abaixo do alvo do artigo (2501). Mantendo todos os 2446 disponíveis.
Amostragem para escala do artigo: 2446 legítimos + 501 phishing = 2947 e-mails (alvo do artigo: 2501+501=3002) [deduplicar_antes=True]
Dataset consolidado: 2947 registros — legítimos: 2446, phishing: 501


# B) DATA PROCESSING
<DISCORRER MAIS>

In [3]:
# --- Módulo B: processamento de texto ---
df = processamento.pipeline_processamento(df)


Pré-processamento concluído: 2947 registros (1 ficaram vazios)


# C) DATABASE BALANCE

In [5]:
# --- Módulo C: balanceamento + treino + teste ---
# 07/10/2026 (Fase 3): undersampling configurável (ver célula de configuração)
## df_balanceado = balanceamento.aplicar_undersampling(df)
df_balanceado = balanceamento.aplicar_undersampling(df, n_por_classe=undersampling_alvo)
df_treino, df_teste = balanceamento.dividir_treino_teste(df_balanceado)

####### Utilizar para avaliar amostragem utilizada
#############inicio debug##############################
#dup_no_balanceado = df_balanceado.duplicated(subset="body_processado", keep=False)
#print("duplicatas no balanceado:", dup_no_balanceado.sum(), "de", len(df_balanceado))
#print(df_balanceado["class_label"].value_counts())

#dup_legit = df_balanceado[df_balanceado["class_label"] == 0].duplicated(subset="body_processado", keep=False)
#dup_phish = df_balanceado[df_balanceado["class_label"] == 1].duplicated(subset="body_processado", keep=False)
#print("duplicatas DENTRO de legítimos:", dup_legit.sum())
#print("duplicatas DENTRO de phishing:", dup_phish.sum())

#sobreposicao = set(df_treino["body_processado"]) & set(df_teste["body_processado"])
#print(f"E-mails processados idênticos entre treino e teste: {len(sobreposicao)}")

#if sobreposicao:
#    afetados_treino = df_treino[df_treino["body_processado"].isin(sobreposicao)]
#    afetados_teste = df_teste[df_teste["body_processado"].isin(sobreposicao)]
#    print("treino:", len(afetados_treino), afetados_treino["class_label"].value_counts().to_dict())
#    print("teste:", len(afetados_teste), afetados_teste["class_label"].value_counts().to_dict())

#raise SystemExit("Checagem de duplicatas concluída — parando aqui antes do treino.")

#######################fim debug###################

# 01/10/2026 (Fase 3): modelo passado como parâmetro + padrão de saída por modelo (ver célula de configuração)
## tokenizer, modelo_tb = balanceamento.carregar_tokenizer_e_modelo()
tokenizer, modelo_pre_treinado = balanceamento.carregar_tokenizer_e_modelo(checkpoint_modelo)
dataset_treino, dataset_teste = balanceamento.tokenizar_dataset(df_treino, df_teste, tokenizer)
## trainer, tempo_treino = balanceamento.treinar_modelo(modelo_tb, tokenizer, dataset_treino, dataset_teste)
trainer, tempo_treino = balanceamento.treinar_modelo(
    modelo_pre_treinado, tokenizer, dataset_treino, dataset_teste, output_dir=pasta_checkpoints
)
predicoes, tempo_teste = balanceamento.testar_modelo(trainer, dataset_teste)
## balanceamento.salvar_modelo(trainer, tokenizer)
balanceamento.salvar_modelo(trainer, tokenizer, caminho=pasta_modelo)

Balanceamento aplicado: {0: 502, 1: 501} (n_por_classe={0: 502, 1: 501})
Treino: {1: 351, 0: 351}
Teste: {0: 151, 1: 150}


Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: huawei-noah/TinyBERT_General_4L_312D
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
fit_denses.{0, 1, 2, 3, 4}.weight          | UNEXPECTED | 
fit_denses.{0, 1, 2, 3, 4}.bias            | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/arch

Map:   0%|          | 0/702 [00:00<?, ? examples/s]

Map:   0%|          | 0/301 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.221937,0.097561,0.973422,0.972789
2,0.115891,0.081885,0.980066,0.979730
3,0.004576,0.045144,0.990033,0.990099
4,0.002334,0.104219,0.976744,0.976271
5,0.001608,0.103376,0.976744,0.976271
6,0.001244,0.071066,0.986711,0.986842
7,0.001055,0.062803,0.990033,0.990099
8,0.000946,0.054884,0.990033,0.990099
9,0.000877,0.052098,0.990033,0.990099
10,0.000856,0.051356,0.990033,0.990099


Tempo de treino: 424.1s [protocolo=artigo, épocas executadas: 10]


Tempo total de inferência no teste: 2.35s


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Modelo salvo em: modelo\tinybert/


In [6]:
import hashlib, glob, os, sys
import torch, transformers

def impressao(df):
    texto = "\n".join(sorted(df["body_processado"].astype(str)))
    return hashlib.md5(texto.encode("utf-8")).hexdigest()

print("balanceado:", impressao(df_balanceado))
print("treino:    ", impressao(df_treino))
print("teste:     ", impressao(df_teste))

arquivos = glob.glob(os.path.join("dados_philder", "spamassassin", "easy_ham", "*"))
print("3 primeiros arquivos do glob:", [os.path.basename(a) for a in arquivos[:3]])
print("python", sys.version.split()[0], "| torch", torch.__version__, "| transformers", transformers.__version__)

balanceado: b5eed2f9bf4f8dd5bd32fe182f56a8c9
treino:     d7ee416586b457c1696481c80eeef4cc
teste:      c2aecb1bdebe9d1fc4b98740e7e08da6
3 primeiros arquivos do glob: ['00001.7c53336b37003a9286aba55d2945844c', '00002.9c4069e25e1ef370c078db7ee85ff9ac', '00003.860e3c3cee1b42ead714c5c874fe25f7']
python 3.12.13 | torch 2.14.0+cpu | transformers 5.16.1


# D) PERFORMANCE ASSESSMENT
## Alterações identificadas:

- [x] Incluir na tabela de métricas o Tempo de treino e o tempo de teste e tempo de detecção
- [x] Equacionar o número de épocas com o do artigo (10 épocas)
- [x] Executar um treinamento com o dataset atual, simulando a quantidade de amostras do artigo original
- [x] Deduplicar dados de dataset de phishing para melhorar qualidade do trainamento e teste
- [x] Deduplicar dados de dataset de phishing de de ham para comparar resultado em relação a deduplicação somente de phishing. 
- [x] Reaplicar análise de resultado conforme artigo (3 rodadas)?
    - Continuar com 6 rodadas [**Fase 3**]

     >To reinforce the consistency and reliability of the reported results, a statistical analysis was conducted across three independent runs for each model using the undersampling approach.
- [x]  Finalizado ajustes rodar notebook no google colabs e comparar os resultados IES, TES, RTDE
      - Executar [**Fase 3**]
- [x] Extensão do treino/teste para os outros modelos do aritgo (função de treino parametrizado [nome do modelo | e parametros esepeíficos])
    - [ ] Iniciar a comparacao - 1 rodada [**Fase 3**]
    - [ ] Comparação - 6 rodadas [**Fase 3**]
- [x] Manter deduplicação dos dados [**Fase 3**]
- [x] Iniciar as partes de implementação do artigo do overleaf e compartilhar com o professor [**Fase 3**]
   - [ ] Escrever artigo em português e abastract em inglês?
- [ ] Realizar comparação com o estudo de ablação (sem pesos sem undersample --> `from_pretrained` para `from_config`)
- [ ] Identificar padrão de unidades usada no artigo para avaliar ajuste e necessidade de conversão de dados, pois os resultados são bem diferentes. Considerar também o hardware utilizado


# Resultados

<details>
<summary>Clique para expandir</summary>

## Resultados do artigo original:

> **Correção 01/10/2026:** os tempos desta tabela estavam com os valores da linha **DistilBERT (2)** da Tabela 3 do artigo (111,83 s / 4,48 s / 14,8841 ms). Substituídos pelos valores da linha **TinyBERT (2)**. As métricas preditivas já estavam corretas.

### Tabela 1 — Métricas de Desempenho Preditivo (PHILDER = TinyBERT (2), execução com undersampling — Tabela 3 do artigo)

| Modelo | Acc. | Prec. | Rec. | F1 | T. Treino (S)| T. Teste (s) | Tempo de Detecção (ms) |
|---|---|---|---|---|---|---|---|
| TinyBERT (2) | 0,9927 | 0,9896 | 0,9960 | 0,9927 | 180,52 | 6,22 | 12,9811 |

### Tabela 2 — Métricas de Eficiência Computacional (PHILDER = TinyBERT + undersampling)
**Valores não normalizados**

| Métrica | Fórmula bruta sobre a Tabela 3 do artigo | Valor exibido nas Figuras 3–5 do artigo |
|---|---|---|
| TES | 0,9927 / 180,52 = **0,0055** | 0,0389 |
| IES | 0,9927 / 6,22 = **0,1596** | 1,0788 |
| RTDE | 0,9927 / 0,0129811 = **76,47** | 324,7750 |

*As Figuras 3–5 não são consistentes com os tempos da Tabela 3 (ex.: TES = 0,0389 implicaria ≈ 25,5 s de treino, valor ausente da Tabela 3). Para comparação usar a coluna da Tabela 3 (dados medidos). Outras inconsistências internas do artigo: o texto cita tempo de teste do TinyBERT undersampling = 2,57 s (Tabela 3: 6,22 s) e "12.9811 s" por amostra (Tabela 3: ms).*

## Resultados deste Trabalho Fase2

### Tabela 1 — Métricas de Desempenho Preditivo

**Data Medição: 17/09/2026**

**Base Nazario 1501 amostras**

| Modelo | Acc. | Prec. | Rec. | F1 |
|---|---|---|---|---|
| philder |0,9936 | 0,9957 | 0,9915 | 0,9936 |

**Data Medição: 19/09/2026**

**Base Nazario 501 amostras**
| Modelo | Acc. | Prec. | Rec. | F1 | T. Treino (S)| T. Teste (s) | Tempo de Detecção (ms) |
|---|---|---|---|---|---|---|---|
| philder |0,9867 | 1,0 | 0,9733 | 0,9865 | 255.8025 | 4.5329 | 15.0595 |

**Data Medição: 21/09/2026**

**Base Nazario 501 amostras**

**Até 10 épocas, + parametros p/ evitar overfiting**

| Modelo | Acc. | Prec. | Rec. | F1 | T. Treino (S)| T. Teste (s) | Tempo de Detecção (ms) |
|---|---|---|---|---|---|---|---|
| philder |0,9933 | 1,0 | 0,9867 | 0,9933 | 205,5063 | 2,1591 | 7,1734 |


**Data Medição: 22/09/2026**

**Base Nazario 501 amostras**

**Até 10 épocas, + parametros p/  evitar overfiting**

**Aplicação de técnica de deduplicação das amostras `df.drop_duplicates`**

| Modelo | Acc. | Prec. | Rec. | F1 | T. Treino (S)| T. Teste (s) | Tempo de Detecção (ms) |
|---|---|---|---|---|---|---|---|
| philder |0,9792 | 0,97992 | 0,9792 | 0,9792 | 298,1438 | 2,0680 | 7,1556 |


**Data Medição: 23/09/2026**

**Base Nazario 501 amostras**

**Até 10 épocas, + parametros p/  evitar overfiting**

**Aplicação de técnica de deduplicação das amostras de mensagens phishing `df.drop_duplicates` ANTES DA CONSOLIDAÇÃO DA BASE PARA TREINAMENTO**

| Modelo | Acc. | Prec. | Rec. | F1 | T. Treino (S)| T. Teste (s) | Tempo de Detecção (ms) |
|---|---|---|---|---|---|---|---|
| philder |0,9900 | 0,9868 | 0,9933 | 0,9900 | 263,5980 | 2,3067 | 7,6633 |


**Data Medição: 24/09/2026**

**Base Nazario 501 amostras**

**Até 10 épocas, + parametros p/  evitar overfiting**

**Aplicação de técnica de deduplicação das amostras de mensagens phishing e Legítimas `df.drop_duplicates` ANTES DA CONSOLIDAÇÃO DA BASE PARA TREINAMENTO**

A precisão volta a ficar suspeita (~1,0). Investigação revela dois bugs reais: (1) a célula do Módulo A no notebook nunca chamava amostrar_quantidade_artigo — usava código residual de 19/09 (random.sample sem deduplicação)


**Data Medição: 25/09/2026**

**Base Nazario 501 amostras**

**Até 10 épocas, + parametros p/  evitar overfiting**

**Aplicação de técnica de **deduplicação** das amostras de mensagens **phishing** `df.drop_duplicates` ANTES DA CONSOLIDAÇÃO DA BASE PARA TREINAMENTO**

**Média 6 rodadas de trainamento**

| Modelo | Acc. | Prec. | Rec. | F1 | T. Treino (S)| T. Teste (s) | Tempo de Detecção (ms) |
|---|---|---|---|---|---|---|---|
| philder |0,9939 | 0,9989 | 0,9989 | 0,9939 | 198,06 | 2,23 | 7,40 |
|Desv Padrão | 0,0033 | 0,0027 | 0,0054 | 0,0033 | 25,02 | 0,16 | 0,55 |

**Data Medição: 25/09/2026**

**Base Nazario 501 amostras**

**Até Até 10 épocas, + parametros p/  evitar overfiting**

**Aplicação de técnica de **deduplicação** das amostras de mensagens **phishing** e **Legítimas** `df.drop_duplicates` ANTES DA CONSOLIDAÇÃO DA BASE PARA TREINAMENTO**

**Média 6 rodadas de trainamento**

| Modelo | Acc. | Prec. | Rec. | F1 | T. Treino (S)| T. Teste (s) | Tempo de Detecção (ms) |
|---|---|---|---|---|---|---|---|
| philder |0,9967 | 0,9978 | 0,9956 | 0,9967 | 352,24 | 3,92 | 13,03 |
|Desv Padrão | 0,0021 | 0,0034 | 0,0034 | 0,0021 | 44,90 | 0,29|  0,96 |


**Data Medição: 01/10/2026**

**INÍCIO FASE 3**

**Base Nazario 501 amostras**

**Até Até 10 épocas, + parametros p/  evitar overfiting**

**Aplicação de técnica de **deduplicação** das amostras de mensagens **phishing** e **Legítimas** `df.drop_duplicates` ANTES DA CONSOLIDAÇÃO DA BASE PARA TREINAMENTO**

**Média 6 rodadas de trainamento**

**Ajuste de hiperparâmetros que lidam com o overfiting**

| Modelo | Acc. | Prec. | Rec. | F1 | T. Treino (S)| T. Teste (s) | Tempo de Detecção (ms) |
|---|---|---|---|---|---|---|---|
| philder (exec. 1) | 0,9934 | 0,9933 | 0,9933 | 0,9933 | 401,06 | 2,22 | 7,36 |
| philder (exec. 2) | 0,9900 | 0,9868 | 0,9933 | 0,9900 | 396,51 | 2,24 | 7,44 |
| philder (exec. 3) | 0,9934 | 0,9933 | 0,9933 | 0,9933 | 397,82 | 2,21 | 7,35 |
| philder (exec. 4) | 0,9934 | 0,9933 | 0,9933 | 0,9933 | 412,32 | 2,26 | 7,49 |
| philder (exec. 5) | 0,9934 | 0,9933 | 0,9933 | 0,9933 | 425,64 | 2,72 | 9,03 |
| philder (exec. 6) | 0,9934 | 0,9933 | 0,9933 | 0,9933 | 405,07 | 2,43 | 8,07 |
| **philder (média)** | **0,9928** | **0,9922** | **0,9933** | **0,9928** | **406,40** | **2,34** | **7,79** |
| Desv. Padrão | 0,0014 | 0,0027 | 0,0000 | 0,0013 | 11,02 | 0,20 | 0,66 |

**Data Medição: 02/10/2026**

**Expansão dos modelos avaliados — piloto (1 execução por modelo)**

**Protocolo "artigo": 10 épocas fixas, sem early stopping e sem seleção de checkpoint no teste; mesmos dados (random_state=42) e hiperparâmetros para todos os modelos**

| Modelo | Acc. | Prec. | Rec. | F1 | T. Treino (S)| T. Teste (s) | Tempo de Detecção (ms) |
|---|---|---|---|---|---|---|---|
| TinyBERT (philder) | 0,9934 | 0,9933 | 0,9933 | 0,9933 | 382,19 | 2,15 | 7,14 |
| MiniLM | 0,9867 | 0,9803 | 0,9933 | 0,9868 | 1430,11 | 9,01 | 29,95 |
| DistilBERT | 0,9967 | 0,9934 | 1,0000 | 0,9967 | 1629,70 | 15,17 | 50,41 |
| MobileBERT | 0,9900 | 0,9868 | 0,9933 | 0,9900 | 1382,11 | 10,24 | 34,02 |
| ALBERT | 0,9701 | 0,9732 | 0,9667 | 0,9699 | 2929,51 | 35,91 | 119,30 |

### Tabela 2 — Métricas de Eficiência Computacional

**Data Medição: 17/09/2026**

**Base Nazario 1501 amostras**

| Métrica | Valor bruto |
|---|---|
| TES | 0,00268 |
| IES | 0,1474 |
| RTDE | 138,3831 |

**Data Medição: 19/09/2026**

**Base Nazario 501 amostras**

| Métrica | Valor bruto |
|---|---|
| TES | 0,0039 |
| IES | 0,2177 |
| RTDE | 65,505 |

**Data Medição: 21/09/2026**

**Base Nazario 501 amostras**

**Até 10 épocas, + parametros p/  evitar overfiting**

| Métrica | Valor bruto |
|---|---|
| TES | 0,0048 |
| IES | 0,4601 |
| RTDE | 138,4772 |

**Data Medição: 22/09/2026**

**Base Nazario 501 amostras**

**Até 10 épocas, + parametros p/  evitar overfiting**

**Aplicação de técnica de deduplicação das amostras de todas as mensagens (ham, phishing) `df.drop_duplicates` DEPOIS DA CONSOLIDAÇÃO DA BASE PARA TREINAMENTO** 

| Métrica | Valor bruto |
|---|---|
| TES | 0,0033 |
| IES | 0,4735 |
| RTDE | 136,8492 |

**Data Medição: 23/09/2026**

**Base Nazario 501 amostras**

**Até 10 épocas, + parametros p/  evitar overfiting**

**Aplicação de técnica de deduplicação das amostras de mensagens phishing `df.drop_duplicates` ANTES DA CONSOLIDAÇÃO DA BASE PARA TREINAMENTO**

| Métrica | Valor bruto |
|---|---|
| TES | 0,00378|
| IES | 0,4292 |
| RTDE | 129,1910 |


**Data Medição: 24/09/2026**

**Base Nazario 501 amostras**

**Até 10 épocas, + parametros p/  evitar overfiting**

**Aplicação de técnica de deduplicação das amostras de mensagens phishing e Legítimas `df.drop_duplicates` ANTES DA CONSOLIDAÇÃO DA BASE PARA TREINAMENTO**

A precisão volta a ficar suspeita (~1,0). Investigação revela dois bugs reais: (1) a célula do Módulo A no notebook nunca chamava amostrar_quantidade_artigo — usava código residual de 19/09 (random.sample sem deduplicação)

**Data Medição: 25/09/2026**

**Base Nazario 501 amostras**

**Até 10 épocas, + parametros p/  evitar overfiting**

**Aplicação de técnica de **deduplicação** das amostras de mensagens **phishing** `df.drop_duplicates` ANTES DA CONSOLIDAÇÃO DA BASE PARA TREINAMENTO**

**Média 6 rodadas de trainamento**

| Métrica | Valor bruto |
|---|---|
| TES | 0,005088 ± 0,000664 |
| IES | 0,447961 ± 0,032449 |
| RTDE | 134,8363 ± 9,7670 |

**Data Medição: 25/09/2026**

**Base Nazario 501 amostras**

**Até 10 épocas, + parametros p/  evitar overfiting**

**Aplicação de técnica de **deduplicação** das amostras de mensagens **phishing** e **ham** `df.drop_duplicates` ANTES DA CONSOLIDAÇÃO DA BASE PARA TREINAMENTO**

**Média 6 rodadas de trainamento**

| Métrica | Valor bruto |
|---|---|
| TES | 0,002872 ± 0,000406 |
| IES | 0,255279 ± 0,018430 |
| RTDE | 76,8516 ± 5,5475 |


**Data Medição: 01/10/2026**

**INÍCIO FASE 3**

**Base Nazario 501 amostras**

**Até Até 10 épocas, + parametros p/  evitar overfiting**

**Aplicação de técnica de **deduplicação** das amostras de mensagens **phishing** e **Legítimas** `df.drop_duplicates` ANTES DA CONSOLIDAÇÃO DA BASE PARA TREINAMENTO**

**Média 6 rodadas de trainamento**

**Ajuste de hiperparâmetros que lidam com o overfiting**

| Métrica | Valor bruto |
|---|---|
| TES | 0,002444 ± 0,000063 |
| TES (líquido) | 0,002590 ± 0,000067 |
| IES | 0,425768 ± 0,032966 |
| RTDE | 128,1561 ± 9,9227 |

**Data Medição: 02/10/2026**

**Expansão dos modelos avaliados — piloto (1 execução por modelo)**

**Protocolo "artigo": 10 épocas fixas, sem early stopping e sem seleção de checkpoint no teste; mesmos dados (random_state=42) e hiperparâmetros para todos os modelos**

| Modelo | TES | TES (líquido) | IES | RTDE |
|---|---|---|---|---|
| TinyBERT (philder) | 0,002599 | 0,002754 | 0,4624 | 139,18 |
| MiniLM | 0,000690 | 0,000737 | 0,1095 | 32,95 |
| DistilBERT | 0,000612 | 0,000675 | 0,0657 | 19,77 |
| MobileBERT | 0,000716 | 0,000774 | 0,0967 | 29,10 |
| ALBERT | 0,000331 | 0,000378 | 0,0270 | 8,13 |

</details>

## Análise Crítica da Avaliação dos Resultados

- Até o momento os resultados de Eficiência são da ordem de 10x piores que o do artigo, indicando que as etapas de treino e testes foram executadoas em um ambiente de poder computacional elevado. O fator mais relevante para a discrepância nos valores foi o tempo. Avaliar este ponto com mais atenção e garantir que as métricas estão realizando o calculo da forma adequada,
- Verificado que, após a equalização da base com o artigo, foi identificado que algumas amostras identicas no dataset treinado incfluenciaram na precisão do modelo (100%). Para contornar a situação foi realizado um tratamento no dataset para a normalização das amostras duplicadas `df.drop_duplicates` e realizada nova medição com este dataset normalizado. Após execução do treinamento o treinamento se mostrou mais consistente,e os dados duplicados causaram interferência nos resultados anteriores.
- Após a ajuste dos dados duplicados, houve mais um ajuste neste sentido, para retirar os dados duplicados da base de phishing (Nazario) antes da consolidação, fazendo, dessa forma, que a base para trainamento e teste tenha 501 amostras consistentes, onde é esperado uma melhora da perforamnce do treino e teste.
    - Foi necessário realizar um debug de código após resultados inconsistêntes de acurácia. Após ajuste os dados passaram a ser consistentes e foram executadas várias rodadas de trinamento (**resultado_final_25092026_estavel[i].csv**)
- A deduplicação foi incluída também na amostra de ham (spamassassin) para verificar se os resultados seriam mais consistentes. Como a técnica do undersample foi aplicada, a deduplicação não influiu na quantidade de amostras usadas no treino e no teste, mas as filtrou, impedindo que amostras duplicadas após o processo de normalzação do corpo do email (rem cabeçalho, links etc..) não fizessem parte do corpo de treino. O resultado consolidado da execução deste processo segue detalhado abaixo:
   
  ```
    Total de e-mails legítimos carregados: 2501
    Total de e-mails de phishing carregados: 1565 (coluna: 'body')
    Pré-processamento concluído: 1565 registros (38 ficaram vazios)
    Deduplicação prévia do pool de 'phishing' (texto processado): 1565 e-mails -> 1472 únicos (93 duplicatas descartadas antes da amostragem)
    Pré-processamento concluído: 2501 registros (2 ficaram vazios)
    Deduplicação prévia do pool de 'legítimos' (texto processado): 2501 e-mails -> 2446 únicos (55 duplicatas descartadas antes da amostragem)
    Aviso: pool único de 'legítimos' tem 2446 e-mails, igual ou abaixo do alvo do artigo (2501). Mantendo todos os 2446 disponíveis.
    Amostragem para escala do artigo: 2446 legítimos + 501 phishing = 2947 e-mails (alvo do artigo: 2501+501=3002) [deduplicar_antes=True]
    Dataset consolidado: 2947 registros — legítimos: 2446, phishing: 501
  ```
  
  - O treinamento foi executado em 6 rodadas novamente (**resultado_final_25092026-3_estavel[i].csv**). 
- Aparentemente os valores estão normalizados (0,1). Avaliar qual técnica foi utilizada (normalização?) para os vaores de métrica de desempenho.
### Fase 3
 - **01/10/2026:** Nesta primeira etapa os resultados foram refinados e encontram-se bem próximos do artigo original, indicando que os hyperparâmetros estão ajustados.
     - Nesta etapa de execução foram ajustados os hiperparâmetros da etapa de balanceamento, forçando o treino para épocas, retirando-se o `early stop` a alterando alguns hyperparametros para os valores default da função:
       | Parâmetro (`treinar_modelo`) | Antes — 25/09(b), protocolo `"selecao"` | Depois — 01/10, protocolo `"artigo"` | Efeito |
       |---|---|---|---|
       | `EarlyStoppingCallback` | `early_stopping_patience=2` | removido (`callbacks=[]`) | Treino sempre roda as 10 épocas |
       | `load_best_model_at_end` | `True` | `False` (padrão) | Modelo final = época 10, não o de melhor F1 no teste |
       | `metric_for_best_model` | `"f1"` | não definido | Sem critério de escolha de checkpoint |
       | `greater_is_better` | `True` | não definido | Idem |
       | `save_strategy` | `"epoch"` | `"no"` | Sem gravação de checkpoint por época (menos I/O no tempo de treino) |
       | `save_total_limit` | `2` | não definido | Sem checkpoints, não há limite a aplicar |
    - Manter padrão para próximas avaliações
    - Foram incluídas valores de médição (tempo líquido) para as métricas de desempenho. Estas métricas devem ser analisadas nas fases seguintes.

 - **02/10/2026:** Após a consolidação do dia 01/10, o notebook foi ajustado para passar por meio de parâmetro o modelo a ser treinado, abrangendo todos os modelos avaliados no artigo. Os hiperparâmetros utilizados no treino foram os mesmos utilizados em todos os modelos. Foi executada 01 rodada de validação (piloto) para cada modelo e computados os resultados para comparação, que já refletiu o encontrado no artgo original.
 - **03/10/2026:** Durante execução no google colabs foi verificado legeira discrepância de dados entre os resultados encontrados offline. Foi identificado que a ordem que o sistema carrega as amostras no colab é deiferente do windows (mesma versão de código). Para manter a padronização dos dados, foi realizada inclusão do parametro `sort` na importação dos dados de ham e os resultados se mantiveram constantes. Execução dos modelos `tinybert` e `minilm` para avaliação de execução em outro ambiente. Os recursos disponibilizados pelo google colabs são menores que o local e isso influi nos resultados de performance

- **08/10/2026:** Execução em lote (6 rodadas)

  
**Tarefas próxima Semana - 20/10**
- Desenvolver implementacão no artigo
- utilizar modelo de tabela do artigo estudo para comparação com os estudos correlatos
- **Ficar TOP  na análise de treinamento de deep learning**
- Próxima prova adiada para o  dia 03/11

In [7]:
# --- Módulo D: avaliação de desempenho ---
metricas = avaliacao.calcular_metricas_primarias(df_teste["class_label"].tolist(), predicoes)
# 01/10/2026 (Fase 3): separa o tempo de avaliação por época do tempo de treino (TES bruto x TES líquido)
## eficiencia = avaliacao.calcular_metricas_eficiencia(metricas["accuracy"], tempo_treino, tempo_teste, len(df_teste))
tempo_aval = avaliacao.calcular_tempo_avaliacao(trainer.state.log_history)
eficiencia = avaliacao.calcular_metricas_eficiencia(
    metricas["accuracy"], tempo_treino, tempo_teste, len(df_teste), tempo_avaliacao=tempo_aval
)
# 01/10/2026 (Fase 3): saída por modelo, com número de rodada automático
## avaliacao.gerar_curva_perda(trainer.state.log_history)
## avaliacao.salvar_resultados({**metricas, **eficiencia})
rodada = proxima_rodada(pasta_resultados, data_execucao)
avaliacao.gerar_curva_perda(
    trainer.state.log_history,
    caminho_imagem=os.path.join(pasta_resultados, f"curva_perda-{rodada}_{data_execucao}.png"),
    titulo=f"Curva de perda — fine-tuning {modelo_escolhido} (rodada {rodada})",
)
avaliacao.salvar_resultados(
    # 07/10/2026: registra com qual undersampling o resultado foi gerado
    {"modelo": modelo_escolhido, "checkpoint": checkpoint_modelo, "rodada": rodada, **metricas, **eficiencia,
     "undersampling": str(undersampling_alvo)},
    caminho_csv=os.path.join(pasta_resultados, f"resultado_final-{rodada}_{data_execucao}.csv"),
)

Curva de perda salva em: resultados\tinybert\curva_perda-2_07102026.png
Resumo salvo em: resultados\tinybert\resultado_final-2_07102026.csv


In [2]:
## TEstatar config especifica albert

# ===== EXPERIMENTO COMPLEMENTAR (09/10/2026) — ALBERT com os hiperparâmetros do artigo original =====
# Fonte: Lan et al. (ALBERT), Tabela 14, tarefa SST-2 -> lr 1e-5, batch 32, warmup ≈ 6% (Análise, Seção 9.6).
# Célula independente: NÃO altera o pacote philder nem as células do protocolo principal.
# Muda só: learning rate, batch de treino e warmup. Mantém: 10 épocas, weight_decay 0.01,
# max_length 128, batch de avaliação 8 (tempo de teste comparável), dados 502/501, semente 42.
import os, math, time, hashlib, datetime
from transformers import Trainer, TrainingArguments
from philder import database_consolidation as consolidacao_exp
from philder import data_processing as processamento_exp
from philder import database_balance as balanceamento_exp
from philder import performance_assessment as avaliacao_exp

EXP_MODELO = "albert"
EXP_CHECKPOINT = "albert-base-v2"
EXP_LR = 1e-5
EXP_BATCH = 32
EXP_EPOCAS = 10
EXP_WARMUP_FRACAO = 0.06
EXP_UNDERSAMPLING = {0: 502, 1: 501}

# --- Dados (mesmo pipeline do lote final) + impressão digital ---
leg = consolidacao_exp.importar_corpus_spamassassin("dados_philder/spamassassin")
phi = consolidacao_exp.importar_corpus_nazario("dados_philder/Nazario.csv")
leg_am, phi_am = consolidacao_exp.amostrar_quantidade_artigo(leg, phi, deduplicar_antes=True)
df_exp = processamento_exp.pipeline_processamento(consolidacao_exp.consolidar_dataset(leg_am, phi_am))
df_bal_exp = balanceamento_exp.aplicar_undersampling(df_exp, n_por_classe=EXP_UNDERSAMPLING)
df_tr_exp, df_te_exp = balanceamento_exp.dividir_treino_teste(df_bal_exp)
_md5 = lambda d: hashlib.md5("\n".join(sorted(d["body_processado"].astype(str))).encode("utf-8")).hexdigest()
hashes_exp = {"hash_balanceado": _md5(df_bal_exp), "hash_treino": _md5(df_tr_exp), "hash_teste": _md5(df_te_exp)}
print(hashes_exp)
# 02/11/2026: hash do teste do lote oficial com máscaras de links e e-mails (Análise, Seção 10.1).
# Até 01/11/2026: "c2aecb1bdebe9d1fc4b98740e7e08da6" (lote de 07/10, sem máscara).
assert hashes_exp["hash_teste"] == "ccfa3c3612628cdfcee16eab995a7d00", "Teste diferente da referência 502/501 — parar e investigar."

# --- Modelo, tokenização e hiperparâmetros ---
tok_exp, mod_exp = balanceamento_exp.carregar_tokenizer_e_modelo(EXP_CHECKPOINT)
ds_tr_exp, ds_te_exp = balanceamento_exp.tokenizar_dataset(df_tr_exp, df_te_exp, tok_exp)
passos_totais = math.ceil(len(ds_tr_exp) / EXP_BATCH) * EXP_EPOCAS
passos_warmup = round(EXP_WARMUP_FRACAO * passos_totais)
print(f"Passos de treino: {passos_totais} | warmup: {passos_warmup} passos ({EXP_WARMUP_FRACAO:.0%})")

pasta_exp = os.path.join("resultados", "experimento_hp", EXP_MODELO)
os.makedirs(pasta_exp, exist_ok=True)
data_exp = datetime.date.today().strftime("%d%m%Y")
rodada_exp = 1 + max([int(f.split("-")[1].split("_")[0]) for f in os.listdir(pasta_exp)
                      if f.startswith("resultado_final-")] or [0])

args_exp = TrainingArguments(
    output_dir=os.path.join("modelo", "experimento_hp", EXP_MODELO, "checkpoints"),
    num_train_epochs=EXP_EPOCAS,
    learning_rate=EXP_LR,
    per_device_train_batch_size=EXP_BATCH,
    per_device_eval_batch_size=8,
    warmup_steps=passos_warmup,
    eval_strategy="epoch",
    save_strategy="no",
    weight_decay=0.01,
    logging_steps=2,          # 23 passos por época com batch 32: registra a perda com mais detalhe
    dataloader_pin_memory=False,
    report_to="none",
)
trainer_exp = Trainer(model=mod_exp, args=args_exp, train_dataset=ds_tr_exp, eval_dataset=ds_te_exp,
                      compute_metrics=balanceamento_exp._metricas_durante_treino)

inicio = time.time()
trainer_exp.train()
tempo_treino_exp = time.time() - inicio
predicoes_exp, tempo_teste_exp = balanceamento_exp.testar_modelo(trainer_exp, ds_te_exp)

# --- Métricas e saídas (pasta separada do protocolo principal) ---
metricas_exp = avaliacao_exp.calcular_metricas_primarias(df_te_exp["class_label"].tolist(), predicoes_exp)
tempo_aval_exp = avaliacao_exp.calcular_tempo_avaliacao(trainer_exp.state.log_history)
eficiencia_exp = avaliacao_exp.calcular_metricas_eficiencia(
    metricas_exp["accuracy"], tempo_treino_exp, tempo_teste_exp, len(df_te_exp), tempo_avaliacao=tempo_aval_exp)
avaliacao_exp.gerar_curva_perda(
    trainer_exp.state.log_history,
    caminho_imagem=os.path.join(pasta_exp, f"curva_perda-{rodada_exp}_{data_exp}.png"),
    titulo=f"Curva de perda — {EXP_MODELO} (lr {EXP_LR:g}, batch {EXP_BATCH}, warmup {EXP_WARMUP_FRACAO:.0%}) — rodada {rodada_exp}")
avaliacao_exp.salvar_resultados(
    {"modelo": EXP_MODELO, "checkpoint": EXP_CHECKPOINT, "rodada": rodada_exp,
     "experimento": "hiperparametros_artigo_original", "learning_rate": EXP_LR, "batch_treino": EXP_BATCH,
     "epocas": EXP_EPOCAS, "warmup_steps": passos_warmup, "passos_totais": passos_totais,
     **metricas_exp, **eficiencia_exp, **hashes_exp, "undersampling": str(EXP_UNDERSAMPLING)},
    caminho_csv=os.path.join(pasta_exp, f"resultado_final-{rodada_exp}_{data_exp}.csv"))
print(f"=== Concluído: accuracy={metricas_exp['accuracy']:.4f} | treino={tempo_treino_exp:.1f}s | teste={tempo_teste_exp:.2f}s ===")

Total de e-mails legítimos carregados: 2501
Total de e-mails de phishing carregados: 1565 (coluna: 'body')
Pré-processamento concluído: 1565 registros (38 ficaram vazios)
Deduplicação prévia do pool de 'phishing' (texto processado): 1565 e-mails -> 1472 únicos (93 duplicatas descartadas antes da amostragem)
Pré-processamento concluído: 2501 registros (2 ficaram vazios)
Deduplicação prévia do pool de 'legítimos' (texto processado): 2501 e-mails -> 2446 únicos (55 duplicatas descartadas antes da amostragem)
Aviso: pool único de 'legítimos' tem 2446 e-mails, igual ou abaixo do alvo do artigo (2501). Mantendo todos os 2446 disponíveis.
Amostragem para escala do artigo: 2446 legítimos + 501 phishing = 2947 e-mails (alvo do artigo: 2501+501=3002) [deduplicar_antes=True]
Dataset consolidado: 2947 registros — legítimos: 2446, phishing: 501
Pré-processamento concluído: 2947 registros (1 ficaram vazios)
Balanceamento aplicado: {0: 502, 1: 501} (n_por_classe={0: 502, 1: 501})
Treino: {1: 351, 0: 

Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

[transformers] AlbertForSequenceClassification LOAD REPORT from: albert-base-v2
Key                          | Status     | 
-----------------------------+------------+-
predictions.LayerNorm.bias   | UNEXPECTED | 
predictions.dense.weight     | UNEXPECTED | 
predictions.LayerNorm.weight | UNEXPECTED | 
predictions.dense.bias       | UNEXPECTED | 
predictions.bias             | UNEXPECTED | 
predictions.decoder.bias     | UNEXPECTED | 
classifier.weight            | MISSING    | 
classifier.bias              | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/702 [00:00<?, ? examples/s]

Map:   0%|          | 0/301 [00:00<?, ? examples/s]

Passos de treino: 220 | warmup: 13 passos (6%)


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.292515,0.301541,0.946844,0.945578
2,0.182300,0.167930,0.943522,0.945338
3,0.048306,0.078044,0.976744,0.976744
4,0.024898,0.070524,0.983389,0.983389
5,0.004178,0.088865,0.973422,0.973684
6,0.002932,0.074939,0.980066,0.980132
7,0.002186,0.087777,0.973422,0.973684
8,0.002392,0.096133,0.973422,0.973684
9,0.001603,0.096762,0.973422,0.973684
10,0.001634,0.095374,0.973422,0.973684


Tempo total de inferência no teste: 40.46s
Curva de perda salva em: resultados\experimento_hp\albert\curva_perda-1_09102026.png
Resumo salvo em: resultados\experimento_hp\albert\resultado_final-1_09102026.csv
=== Concluído: accuracy=0.9734 | treino=3010.5s | teste=40.46s ===


In [4]:
# ===== CÉLULA TEMPORÁRIA (01/11/2026) — evidência da deduplicação pelo texto processado =====
# Mostra, para cada rótulo, 2 e-mails BRUTOS diferentes que viram o MESMO body_processado
# após o Módulo B — exatamente o critério usado em amostrar_quantidade_artigo(deduplicar_antes=True).
# Usa brutos_legitimos / brutos_phishing já carregados na célula do Módulo A. Apenas exibe; pode ser apagada.
import difflib
import pandas as pd

LIMITE = 600   # caracteres exibidos de cada texto

def _grupos_duplicados(emails_brutos):
    campos = [consolidacao.extrair_campos_email(m) for m in emails_brutos]
    df_tmp = processamento.pipeline_processamento(pd.DataFrame(campos))
    df_tmp["bruto"] = emails_brutos
    dup = df_tmp[df_tmp.duplicated(subset="body_processado", keep=False)]
    grupos = list(dup.groupby("body_processado", sort=False))
    # separa grupos em que os brutos são idênticos dos que só coincidem DEPOIS do processamento
    so_apos = [(t, g) for t, g in grupos if g["bruto"].nunique() > 1]
    identicos = [(t, g) for t, g in grupos if g["bruto"].nunique() == 1]
    return df_tmp, so_apos, identicos

def _mostrar(texto):
    texto = texto.replace("\r", "\n")
    return texto[:LIMITE] + (f" [...] (+{len(texto) - LIMITE} caracteres)" if len(texto) > LIMITE else "")

def _onde_diferem(a, b, contexto=60):
    a, b = a.replace("\r", "\n"), b.replace("\r", "\n")
    blocos = [op for op in difflib.SequenceMatcher(None, a, b, autojunk=False).get_opcodes() if op[0] != "equal"]
    for tag, i1, i2, j1, j2 in blocos[:3]:
        print(f"  • {tag}: A[{i1}:{i2}] = {a[i1:i2][:contexto]!r}  |  B[{j1}:{j2}] = {b[j1:j2][:contexto]!r}")
    if len(blocos) > 3:
        print(f"  • ... e mais {len(blocos) - 3} trecho(s) diferente(s)")

def evidenciar(emails_brutos, nome):
    df_tmp, so_apos, identicos = _grupos_duplicados(emails_brutos)
    removidos = df_tmp["body_processado"].duplicated(keep="first").sum()
    print("#" * 95)
    print(f"{nome}: {len(emails_brutos)} e-mails brutos -> {len(emails_brutos) - removidos} únicos ({removidos} descartados)")
    print(f"  grupos de duplicatas: {len(so_apos) + len(identicos)} "
          f"({len(identicos)} com texto bruto idêntico; {len(so_apos)} só iguais APÓS o processamento)")
    print("#" * 95)
    if not so_apos:
        print("Nenhum par diferente no bruto e igual após o processamento.\n")
        return
    texto_proc, g = max(so_apos, key=lambda tg: len(tg[0]))   # o par com mais conteúdo, para ficar ilustrativo
    a, b = g["bruto"].iloc[0], g["bruto"].iloc[1]
    print(f"\n--- E-mail A (bruto, posição {g.index[0]} na lista, {len(a)} caracteres) ---\n{_mostrar(a)}")
    print(f"\n--- E-mail B (bruto, posição {g.index[1]} na lista, {len(b)} caracteres) ---\n{_mostrar(b)}")
    print("\n--- Onde os brutos diferem ---")
    _onde_diferem(a, b)
    print(f"\n--- body_processado (IGUAL nos dois; {len(texto_proc)} caracteres) ---\n{_mostrar(texto_proc)}")
    print(f"\nResultado: só o 1º ({g.index[0]}) fica no pool; o(s) outro(s) {list(g.index[1:])} é(são) descartado(s).\n")

evidenciar(brutos_phishing, "PHISHING (Nazario)")
evidenciar(brutos_legitimos, "HAM / LEGÍTIMO (SpamAssassin)")

Pré-processamento concluído: 1565 registros (38 ficaram vazios)
###############################################################################################
PHISHING (Nazario): 1565 e-mails brutos -> 1471 únicos (94 descartados)
  grupos de duplicatas: 49 (0 com texto bruto idêntico; 49 só iguais APÓS o processamento)
###############################################################################################

--- E-mail A (bruto, posição 185 na lista, 2774 caracteres) ---
Your email ( jose@monkey.org ) is almost full.  Current size       1989MB	  2000MB  Maximum size Your mailbox is running out of data storage and it might be close kindly to add more MB to your mailbox.            Upgrade Now    If don't upgrade your mail box, you will not receive more emails frequently  You will be finding it hard receiving emails.  Your account might be closed  Your important mail will be wiped away   Note: Upgrade if free of charge, you are not charge any service fee.  Copyright ? 2015 Email 